This notebook provides the win probability code for gathering the data

In [2]:
# import library
import streamlit as st
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import sportsdataverse as sdv

In [5]:
# Import college football data from 2025
cfb_data = sdv.cfb.load_cfb_pbp([2025], return_as_pandas=False)

# Convert the polar files to pandas dataframes
cfb_data = cfb_data.to_pandas(use_pyarrow_extension_array=False)

Filter for fbs teams only

In [6]:
# Create a list of all FBS schools
fbs_teams = ["Boston College", "California", "Clemson", "Duke", "Florida State",
    "Georgia Tech", "Louisville", "Miami", "NC State", "North Carolina",
    "Pittsburgh", "SMU", "Stanford", "Syracuse", "Virginia",
    "Virginia Tech", "Wake Forest", "Illinois", "Indiana", "Iowa", "Maryland", "Michigan",
    "Michigan State", "Minnesota", "Nebraska", "Northwestern", "Ohio State",
    "Oregon", "Penn State", "Purdue", "Rutgers", "UCLA",
    "USC", "Washington", "Wisconsin", "Arizona", "Arizona State", "Baylor", "BYU", "Cincinnati",
    "Colorado", "Houston", "Iowa State", "Kansas", "Kansas State",
    "Oklahoma State", "TCU", "Texas Tech", "UCF", "Utah",
    "West Virginia", "Alabama", "Arkansas", "Auburn", "Florida", "Georgia",
    "Kentucky", "LSU", "Mississippi State", "Missouri", "Oklahoma",
    "Ole Miss", "South Carolina", "Tennessee", "Texas", "Texas A&M",
    "Vanderbilt", "Army", "Charlotte", "East Carolina", "Florida Atlantic", "Memphis",
    "Navy", "North Texas", "Rice", "Temple", "Tulane",
    "Tulsa", "UAB", "South Florida", "UTSA", "Boise State", "Colorado State", "Fresno State",
    "Oregon State", "San Diego State", "Texas State", "Utah State", "Washington State",
    "Air Force", "Hawai'i", "Nevada", "New Mexico", "North Dakota State",
    "Northern Illinois", "San José State", "UNLV", "UTEP", "Wyoming",
    "Akron", "Ball State", "Bowling Green", "Buffalo", "Central Michigan",
    "Eastern Michigan", "Kent State", "Miami (OH)", "Ohio", "Sacramento State",
    "Toledo", "Massachusetts", "Western Michigan",
    "Delaware", "Florida International", "Jacksonville State", "Kennesaw State", "Liberty",
    "Middle Tennessee", "Missouri State", "New Mexico State", "Sam Houston", "Western Kentucky",
    "Appalachian State", "Arkansas State", "Coastal Carolina", "Georgia Southern", "Georgia State",
    "James Madison", "Louisiana", "Louisiana Tech", "Marshall", "Old Dominion",
    "South Alabama", "Southern Miss", "Troy", "UL Monroe", "Notre Dame", "UConn"
]

# Now, let's filter the data so that it only keeps fbs vs fbs games
# and it only keeps the regular season games and plays that we care about on offense.
cfb_data = cfb_data[(cfb_data['homeTeamName'].isin(fbs_teams)) & (cfb_data['awayTeamName'].isin(fbs_teams))
].copy()

In [7]:
# Create a dictionary to map the values
school_mapping = {
    "Boston College Eagles": "Boston College",
    "California Golden Bears": "California",
    "Clemson Tigers": "Clemson",
    "Duke Blue Devils": "Duke",
    "Florida State Seminoles": "Florida State",
    "Georgia Tech Yellow Jackets": "Georgia Tech",
    "Louisville Cardinals": "Louisville",
    "Miami Hurricanes": "Miami",
    "NC State Wolfpack": "NC State",
    "North Carolina Tar Heels": "North Carolina",
    "Pittsburgh Panthers": "Pittsburgh",
    "SMU Mustangs": "SMU",
    "Stanford Cardinal": "Stanford",
    "Syracuse Orange": "Syracuse",
    "Virginia Cavaliers": "Virginia",
    "Virginia Tech Hokies": "Virginia Tech",
    "Wake Forest Demon Deacons": "Wake Forest",
    "Illinois Fighting Illini": "Illinois",
    "Indiana Hoosiers": "Indiana",
    "Iowa Hawkeyes": "Iowa",
    "Maryland Terrapins": "Maryland",
    "Michigan Wolverines": "Michigan",
    "Michigan State Spartans": "Michigan State",
    "Minnesota Golden Gophers": "Minnesota",
    "Nebraska Cornhuskers": "Nebraska",
    "Northwestern Wildcats": "Northwestern",
    "Ohio State Buckeyes": "Ohio State",
    "Oregon Ducks": "Oregon",
    "Penn State Nittany Lions": "Penn State",
    "Purdue Boilermakers": "Purdue",
    "Rutgers Scarlet Knights": "Rutgers",
    "UCLA Bruins": "UCLA",
    "USC Trojans": "USC",
    "Washington Huskies": "Washington",
    "Wisconsin Badgers": "Wisconsin",
    "Arizona Wildcats": "Arizona",
    "Arizona State Sun Devils": "Arizona State",
    "Baylor Bears": "Baylor",
    "BYU Cougars": "BYU",
    "Cincinnati Bearcats": "Cincinnati",
    "Colorado Buffaloes": "Colorado",
    "Houston Cougars": "Houston",
    "Iowa State Cyclones": "Iowa State",
    "Kansas Jayhawks": "Kansas",
    "Kansas State Wildcats": "Kansas State",
    "Oklahoma State Cowboys": "Oklahoma State",
    "TCU Horned Frogs": "TCU",
    "Texas Tech Red Raiders": "Texas Tech",
    "UCF Knights": "UCF",
    "Utah Utes": "Utah",
    "West Virginia Mountaineers": "West Virginia",
    "Alabama Crimson Tide": "Alabama",
    "Arkansas Razorbacks": "Arkansas",
    "Auburn Tigers": "Auburn",
    "Florida Gators": "Florida",
    "Georgia Bulldogs": "Georgia",
    "Kentucky Wildcats": "Kentucky",
    "LSU Tigers": "LSU",
    "Mississippi State Bulldogs": "Mississippi State",
    "Missouri Tigers": "Missouri",
    "Oklahoma Sooners": "Oklahoma",
    "Ole Miss Rebels": "Ole Miss",
    "South Carolina Gamecocks": "South Carolina",
    "Tennessee Volunteers": "Tennessee",
    "Texas Longhorns": "Texas",
    "Texas A&M Aggies": "Texas A&M",
    "Vanderbilt Commodores": "Vanderbilt",
    "Army Black Knights": "Army",
    "Charlotte 49ers": "Charlotte",
    "East Carolina Pirates": "East Carolina",
    "Florida Atlantic Owls": "Florida Atlantic",
    "Memphis Tigers": "Memphis",
    "Navy Midshipmen": "Navy",
    "North Texas Mean Green": "North Texas",
    "Rice Owls": "Rice",
    "Temple Owls": "Temple",
    "Tulane Green Wave": "Tulane",
    "Tulsa Golden Hurricane": "Tulsa",
    "UAB Blazers": "UAB",
    "South Florida Bulls": "South Florida",
    "UTSA Roadrunners": "UTSA",
    "Boise State Broncos": "Boise State",
    "Colorado State Rams": "Colorado State",
    "Fresno State Bulldogs": "Fresno State",
    "Oregon State Beavers": "Oregon State",
    "San Diego State Aztecs": "San Diego State",
    "Texas State Bobcats": "Texas State",
    "Utah State Aggies": "Utah State",
    "Washington State Cougars": "Washington State",
    "Air Force Falcons": "Air Force",
    "Hawai'i Rainbow Warriors": "Hawai'i",
    "Nevada Wolf Pack": "Nevada",
    "New Mexico Lobos": "New Mexico",
    "North Dakota State Bison": "North Dakota State",
    "Northern Illinois Huskies": "Northern Illinois",
    "San José State Spartans": "San José State",
    "UNLV Rebels": "UNLV",
    "UTEP Miners": "UTEP",
    "Wyoming Cowboys": "Wyoming",
    "Akron Zips": "Akron",
    "Ball State Cardinals": "Ball State",
    "Bowling Green Falcons": "Bowling Green",
    "Buffalo Bulls": "Buffalo",
    "Central Michigan Chippewas": "Central Michigan",
    "Eastern Michigan Eagles": "Eastern Michigan",
    "Kent State Golden Flashes": "Kent State",
    "Miami (OH) RedHawks": "Miami (OH)",
    "Ohio Bobcats": "Ohio",
    "Sacramento State Hornets": "Sacramento State",
    "Toledo Rockets": "Toledo",
    "Massachusetts Minutemen": "Massachusetts",
    "Western Michigan Broncos": "Western Michigan",
    "Delaware Blue Hens": "Delaware",
    "Florida International Panthers": "Florida International",
    "Jacksonville State Gamecocks": "Jacksonville State",
    "Kennesaw State Owls": "Kennesaw State",
    "Liberty Flames": "Liberty",
    "Middle Tennessee Blue Raiders": "Middle Tennessee",
    "Missouri State Bears": "Missouri State",
    "New Mexico State Aggies": "New Mexico State",
    "Sam Houston Bearkats": "Sam Houston",
    "Western Kentucky Hilltoppers": "Western Kentucky",
    "Appalachian State Mountaineers": "Appalachian State",
    "Arkansas State Red Wolves": "Arkansas State",
    "Coastal Carolina Chanticleers": "Coastal Carolina",
    "Georgia Southern Eagles": "Georgia Southern",
    "Georgia State Panthers": "Georgia State",
    "James Madison Dukes": "James Madison",
    "Louisiana Ragin' Cajuns": "Louisiana",
    "Louisiana Tech Bulldogs": "Louisiana Tech",
    "Marshall Thundering Herd": "Marshall",
    "Old Dominion Monarchs": "Old Dominion",
    "South Alabama Jaguars": "South Alabama",
    "Southern Miss Golden Eagles": "Southern Miss",
    "Troy Trojans": "Troy",
    "UL Monroe Warhawks": "UL Monroe",
    "Notre Dame Fighting Irish": "Notre Dame",
    "UConn Huskies": "UConn",
}

# Map the values to replace the values in pos_team and def_pos_team
cfb_data['pos_team'] = cfb_data['pos_team'].map(school_mapping)
cfb_data['def_pos_team'] = cfb_data['def_pos_team'].map(school_mapping)

# Check that it worked as expected
cfb_data['pos_team'].value_counts()

pos_team
Miami               1298
Ole Miss            1286
Indiana             1271
Virginia            1226
Oregon              1215
                    ... 
Florida              895
Kansas State         854
Kent State           847
Charlotte            825
Sacramento State      84
Name: count, Length: 136, dtype: int64

Plan: Gather the win probabilities after each play

In [ ]:
# Check the data
cfb_data

,season,game_id,game_play_number,pos_team_id,pos_team,def_pos_team_id,def_pos_team,pos_team_score,def_pos_team_score,half,...,go_boost,go_wp_diff,fg_wp_diff,punt_wp_diff,fourth_down_recommendation,two_pt_wp,xp_wp,prob_2pt,two_pt_recommendation,two_pt_wp_diff
4022,2025,401756846,1,66,Iowa State,2306,Kansas State,0,0,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
4023,2025,401756846,2,66,Iowa State,2306,Kansas State,0,0,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
4024,2025,401756846,3,66,Iowa State,2306,Kansas State,0,0,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
4025,2025,401756846,4,66,Iowa State,2306,Kansas State,0,0,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
4026,2025,401756846,5,66,Iowa State,2306,Kansas State,0,0,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166048,2025,401769076,170,84,Indiana,2390,Miami,27,21,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
166049,2025,401769076,171,84,Indiana,2390,Miami,27,21,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
166050,2025,401769076,172,84,Indiana,2390,Miami,27,21,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
166051,2025,401769076,173,84,Indiana,2390,Miami,27,21,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN


In [ ]:
# Round the win probabilities to a percentage
cfb_data['home_wp_after'] = cfb_data['home_wp_after']*100
cfb_data['away_wp_after'] = cfb_data['away_wp_after']*100

export the data

In [ ]:
# Gather only the needed columns for export
cfb_data_export = cfb_data[['game_id','game_play_number','homeTeamName', 'awayTeamName', 'home_wp_after','away_wp_after']]

# export the data
cfb_data_export.to_csv('cfb_2025_game_data.csv', index=False)

Test a line chart

In [32]:
# plot win probability throughout the game for a given game id
game_id = 401756846

cfb_data_game = cfb_data[cfb_data['game_id'] == game_id]

plt.plot(cfb_data_game['game_play_number'], cfb_data_game['home_wp_after'], label=cfb_data_game['homeTeamName'].iloc[0])
plt.plot(cfb_data_game['game_play_number'], cfb_data_game['away_wp_after'], label=cfb_data_game['awayTeamName'].iloc[0])
plt.xlabel('Game Play Number')
plt.ylabel('Win Probability')
plt.title(f'Win Probability for Game {game_id}')
plt.legend()
plt.show()